In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import csv
from datetime import datetime
import time
import re
import os

# ✅ 스크롤 함수
def scroll_to_load_all(driver):
    prev_height = 0
    while True:
        driver.execute_script("window.scrollBy(0, 2000);")
        time.sleep(1.5)
        curr_height = driver.execute_script("return document.body.scrollHeight")
        if curr_height == prev_height:
            break
        prev_height = curr_height

# ✅ 리뷰 수집 함수 (새 탭 사용)
def collect_reviews(driver, restaurant_id, restaurant_name):
    # 현재 창의 핸들 저장
    original_window = driver.current_window_handle
    
    # 새 탭 열기
    driver.execute_script("window.open('');")
    driver.switch_to.window(driver.window_handles[-1])
    
    try:
        url = f"https://pcmap.place.naver.com/restaurant/{restaurant_id}/review/visitor"
        driver.get(url)
        time.sleep(2)

        # 더보기 버튼 클릭하여 모든 리뷰 로드
        while True:
            try:
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(1)
                more_button = WebDriverWait(driver, 2).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, 'a.fvwqf'))
                )
                driver.execute_script("arguments[0].click();", more_button)
                time.sleep(2)
            except:
                break

        reviews = driver.find_elements(By.CSS_SELECTOR, 'li.place_apply_pui.EjjAW')
        print(f"[{restaurant_name}] 리뷰 개수: {len(reviews)}")

        # 파일명 안전하게 생성
        safe_name = re.sub(r'[^\w가-힣]', '_', restaurant_name)[:30]
        now = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        os.makedirs("collected_reviews", exist_ok=True)
        file_name = f"collected_reviews/{safe_name}_{restaurant_id}_{now}.csv"

        with open(file_name, mode='w', newline='', encoding='utf-8-sig') as f:
            writer = csv.writer(f)
            writer.writerow(["식당명", "작성일", "방문횟수", "리뷰 내용", "리뷰 태그"])

            for r in reviews:
                try:
                    date = r.find_element(By.CSS_SELECTOR, 'span.pui__gfuUIT > time').text.strip()
                    
                    # 방문횟수 추출
                    revisit_elements = r.find_elements(By.CSS_SELECTOR, 'span.pui__gfuUIT')
                    revisit = ''
                    try:
                        for elem in revisit_elements:
                            if "번째 방문" in elem.text:
                                revisit = int(elem.text.replace("번째 방문", "").strip())
                                break
                    except:
                        revisit = ''

                    # 더보기 버튼 클릭
                    try:
                        showmore = r.find_element(By.CSS_SELECTOR, 'a[data-pui-click-code="rvshowmore"]')
                        driver.execute_script("arguments[0].click();", showmore)
                        time.sleep(0.2)
                    except:
                        pass

                    content = r.find_element(By.CSS_SELECTOR, 'div.pui__vn15t2').text.strip()

                    # 태그 추출
                    try:
                        tag_area = r.find_element(By.CSS_SELECTOR, 'div.pui__HLNvmI')
                        tags = [tag.text.strip() for tag in tag_area.find_elements(By.CSS_SELECTOR, 'span.pui__jhpEyP')]
                        tag_str = ", ".join(tags)
                    except:
                        tag_str = ''

                    writer.writerow([restaurant_name, date, revisit, content, tag_str])
                except Exception as e:
                    print("❌ 리뷰 수집 오류:", e)

        print(f"✅ 저장 완료: {file_name}")
        
    except Exception as e:
        print(f"❌ 리뷰 수집 실패 ({restaurant_name}): {e}")
    
    finally:
        # 새 탭 닫고 원래 탭으로 돌아가기
        driver.close()
        driver.switch_to.window(original_window)
        print("✅ 원래 검색 페이지로 복귀\n")

# ✅ 메인 크롤링
search_url = "https://map.naver.com/p/search/건대입구역 맛집"
driver = webdriver.Chrome()
driver.get(search_url)
time.sleep(4)

page = 1

while True:
    print(f"📄 {page} 페이지 처리 중...")

    try:
        # searchIframe으로 전환
        driver.switch_to.default_content()
        WebDriverWait(driver, 10).until(EC.frame_to_be_available_and_switch_to_it((By.ID, "searchIframe")))
        scroll_to_load_all(driver)
    except Exception as e:
        print(f"❌ searchIframe 진입 실패: {e}")
        break

    # 식당 링크 가져오기
    place_links = driver.find_elements(By.CSS_SELECTOR, 'a.place_bluelink')
    print(f"링크 수: {len(place_links)}")

    # 각 식당 처리
    for index, link in enumerate(place_links):
        try:
            name = link.text.strip()
            print(f"[🔎] {index+1}번째: {name} 클릭 중...")

            # 식당 클릭
            driver.execute_script("arguments[0].click();", link)
            time.sleep(3)

            # entryIframe에서 ID 추출
            driver.switch_to.default_content()
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "entryIframe")))
            iframe = driver.find_element(By.ID, "entryIframe")
            src = iframe.get_attribute("src")
            
            if "/place/" in src:
                rid = src.split("/place/")[1].split("?")[0]
                if rid.isdigit():
                    print(f"✅ ID 추출 성공: {name} → {rid}")
                    # 리뷰 수집 (새 탭에서)
                    collect_reviews(driver, rid, name)
                else:
                    print(f"❌ 잘못된 ID 형식: {rid}")
            else:
                print(f"❌ ID 추출 실패: {name}")
                
        except Exception as e:
            print(f"⚠️ 오류 발생 (index={index}): {e}")
        
        # 검색 페이지로 돌아가기
        try:
            driver.get(search_url)
            time.sleep(3)
            
            # 현재 페이지로 이동
            if page > 1:
                for _ in range(page - 1):
                    try:
                        driver.switch_to.default_content()
                        WebDriverWait(driver, 5).until(EC.frame_to_be_available_and_switch_to_it((By.ID, "searchIframe")))
                        next_btn = WebDriverWait(driver, 5).until(
                            EC.element_to_be_clickable((By.CSS_SELECTOR, 'a.eUTV2[aria-disabled="false"]'))
                        )
                        driver.execute_script("arguments[0].click();", next_btn)
                        time.sleep(2)
                    except:
                        break
            
            # searchIframe으로 재진입
            driver.switch_to.default_content()
            WebDriverWait(driver, 10).until(EC.frame_to_be_available_and_switch_to_it((By.ID, "searchIframe")))
            scroll_to_load_all(driver)
            # 링크 재수집
            place_links = driver.find_elements(By.CSS_SELECTOR, 'a.place_bluelink')
            
        except Exception as e:
            print(f"❌ 검색 페이지 복귀 실패: {e}")
            break

    # 다음 페이지로 이동
    try:
        driver.switch_to.default_content()
        WebDriverWait(driver, 10).until(EC.frame_to_be_available_and_switch_to_it((By.ID, "searchIframe")))
        next_btn = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, 'a.eUTV2[aria-disabled="false"]'))
        )
        driver.execute_script("arguments[0].click();", next_btn)
        time.sleep(3)
        page += 1
    except Exception as e:
        print(f"🚫 다음 페이지 없음 또는 클릭 실패: {e}")
        break

driver.quit()
print("🎉 전체 리뷰 수집 및 저장 완료")

📄 1 페이지 처리 중...
링크 수: 10
[🔎] 1번째: 건대 숯불부자곱창
예약
톡톡
곱창,막창,양 클릭 중...
✅ ID 추출 성공: 건대 숯불부자곱창
예약
톡톡
곱창,막창,양 → 1261821265
[건대 숯불부자곱창
예약
톡톡
곱창,막창,양] 리뷰 개수: 1553
✅ 저장 완료: collected_reviews/건대_숯불부자곱창_예약_톡톡_곱창_막창_양_1261821265_2025-06-06_21-12-00.csv
✅ 원래 검색 페이지로 복귀

⚠️ 오류 발생 (index=1): Message: no such element: element not found
  (Session info: chrome=137.0.7151.69); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
0   chromedriver                        0x0000000104e22654 cxxbridge1$str$ptr + 2723108
1   chromedriver                        0x0000000104e1a8c8 cxxbridge1$str$ptr + 2690968
2   chromedriver                        0x000000010496e714 cxxbridge1$string$len + 90428
3   chromedriver                        0x000000010497f14c cxxbridge1$string$len + 158580
4   chromedriver                        0x000000010497e228 cxxbridge1$string$len + 154704
5   chromedriver                      